In [1]:
import torch 
print(torch.cuda.is_available())
print(torch.__version__)

True
2.6.0+cu124


In [2]:
!nvidia-smi

Sun Mar  1 23:40:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA TITAN Xp                Off |   00000000:02:00.0  On |                  N/A |
| 23%   33C    P8             17W /  250W |     279MiB /  12288MiB |     31%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Load the validation and test dataset

In [7]:
import pandas as pd 
import numpy as np 
from matplotlib import pyplot as plt
from collections import Counter

df = pd.read_csv('../data/dontpatronizeme_pcl.tsv', sep='\t')

# Remove rows with NA labels 
df = df.dropna() 

# Add a bool_labels column for binary classification
df["bool_labels"] = df["label"] > 1   # is PCL if >1
val_labels = pd.read_csv('../data/dev_semeval_parids-labels.csv')["par_id"]  # extract the labels for val dataset 
df_val = df[df["par_id"].isin(val_labels)].reset_index() 

# read the test dataset 
df_test = pd.read_csv('../data/task4_test.tsv', sep='\t')


# Text Cleaning

In [8]:
# Remove special characters
SPECIAL_CHARACTERS = ['&amp;', '&lt;', '&gt;', '<h>', '\n', '\t']
for char in SPECIAL_CHARACTERS:
    df_val["text"] = df_val["text"].str.replace(char, "")
    df_test["text"] = df_test["text"].str.replace(char, "")

# Replace numbers with 0
df_val["text"] = df_val["text"].str.replace(r"\d+", "0", regex=True)
df_test["text"] = df_test["text"].str.replace(r"\d+", "0", regex=True)


# Add contextual information to the text tokens

In [11]:
def add_info(df): 
    # Append the keyword and country code to the text, and separate them with additional separator tokens
    # Remove dashes in the keyword to match the format in the texts 
    return df["keyword"].str.replace('-', " ") + "<SEP>" + df["country_code"] + "<SEP>" + df["text"]

# Tokenization

In [12]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification, BertTokenizer, BertForSequenceClassification, AutoConfig, Trainer, TrainingArguments

tokenizer_roberta = RobertaTokenizer.from_pretrained("roberta-base") 
tokenizer_bert = BertTokenizer.from_pretrained("bert-base-uncased") 

# define special tokens for separating text 
special_tokens = {"additional_special_tokens": ["<SEP>"]}
tokenizer_roberta.add_special_tokens(special_tokens) 
tokenizer_bert.add_special_tokens(special_tokens) 

# Create text with contextual information 
def tokenize(tokenizer, df): 
    text_with_context = add_info(df) 

    encoding = tokenizer(
        text_with_context.tolist(), 
        padding="max_length",   # Add padding to shorter sentences 
        max_length=256,
        truncation = True, 
        return_attention_mask = True 
    )

    return encoding

/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/_distutils_hack/__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprec

# Convert to pyTorch dataset

In [ ]:
import torch 
from torch.utils.data import DataLoader, TensorDataset
from datasets import Dataset

def to_dataset_val(tokenizer, df): 
    # Obtain tokens (input_ids, attention_mask) from the dataset 
    encoding = tokenize(tokenizer, df) 

    # Return huggingface dataset 
    return Dataset.from_dict({
        "input_ids": encoding["input_ids"], 
        "attention_mask": encoding["attention_mask"], 
        "label": df["label"].values 
    })

# the test set does not have labels 
def to_dataset_test(tokenizer, df): 
    # Obtain tokens (input_ids, attention_mask) from the dataset 
    encoding = tokenize(tokenizer, df) 

    # Return huggingface dataset 
    return Dataset.from_dict({
        "input_ids": encoding["input_ids"], 
        "attention_mask": encoding["attention_mask"]
    })

In [16]:
val_dataset_r = to_dataset_val(tokenizer_roberta, df_val) 
val_dataset_b = to_dataset_val(tokenizer_bert, df_val) 
test_dataset_r = to_dataset_test(tokenizer_roberta, df_test) 
test_dataset_b = to_dataset_test(tokenizer_bert, df_test) 

# set to torch format 
val_dataset_r.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset_b.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset_r.set_format("torch", columns=["input_ids", "attention_mask"])
test_dataset_b.set_format("torch", columns=["input_ids", "attention_mask"])

# Load both BERT and RoBERTa models
They were trained on data formatted the same way: 
- Text cleaning
- Oversample the minority class for each keyword
- Add contextual information including the keyword and the country code to the text 

In [17]:
import os 

# must pass abs path here 
bert_model = BertForSequenceClassification.from_pretrained(os.path.abspath("/vol/bitbucket/bm1325/NLP-PCL-Classification/models/model_bert_ensemble"), local_files_only = True)
roberta_model = RobertaForSequenceClassification.from_pretrained(os.path.abspath("/vol/bitbucket/bm1325/NLP-PCL-Classification/models/model_roberta_ensemble"), local_files_only = True)

trainer_bert = Trainer(model = bert_model)
trainer_roberta = Trainer(model = roberta_model)

# Grid search on the validation dataset

In [ ]:
# convert loaded models to Trainer objects 
trainer_r = Trainer(model = roberta_model)
trainer_b = Trainer(model = bert_model)

# Make predictions on the validation dataset using the roberta model (r) and bert model (b) respectively 
y_pred_r = trainer_r.predict(val_dataset_r) 
y_pred_b = trainer_b.predict(val_dataset_b)

  0%|          | 0/262 [00:00<?, ?it/s]

  0%|          | 0/262 [00:00<?, ?it/s]

In [ ]:
from sklearn.metrics import f1_score

y_true = df_val["bool_labels"]

# x refers to the weight of the roberta model 
x_values = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
best_x = -1 
cur_f1 = 0 
best_pred = None 

for x in x_values: 
    y_pred = y_pred_r.predictions*x + y_pred_b.predictions*(1-x) 

    # Extract the most likely output 
    y_pred = y_pred.argmax(axis=1) 

    # Convert that to boolean values
    y_pred = y_pred>1   # >1 = is PCL 

    # Calculate f1 score 
    f1 = f1_score(y_pred, y_true)
    if f1 > cur_f1: 
        cur_f1 = f1 
        best_x = x 
        best_pred = y_pred

best_x

# Write best_preds to dev.txt

# Use the best x value to make predictions on the test set